# Paper Figure 2 — empirical PCA / MS-AA benchmark

This is the **main-text figure notebook**.

The main figure now focuses only on the direct empirical benchmark:

- PCA reconstruction
- spatial MS-AA
- temporal MS-AA
- raw HTFA reference
- chance

Conditions remain encoded exactly as elsewhere in the manuscript:

- **intact = purple**
- **word-scrambled = green**
- **rest = black**

Method is encoded by line style and marker shape rather than by color.

The projected-vs-reconstructed PCA validation has been moved to a separate
supplementary figure notebook so the main text can focus on the scientifically
important PCA-vs-MS-AA comparison.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

CONDITIONS = ["intact", "word", "rest"]
CONDITION_LABELS = {
    "intact": "Intact narrative",
    "word": "Word-scrambled",
    "rest": "Rest",
}
CONDITION_COLORS = {
    "intact": "#7B3294",
    "word": "#2CA25F",
    "rest": "#222222",
}

T = 300
V = 700

PCA_DIR = Path("pca_aa_equivalence_outputs")
RAW_CSV = PCA_DIR / "raw_htfa_summary.csv"
PCA_CSV = PCA_DIR / "pca_summary.csv"

SPATIAL_FULL_CSV = Path(
    "msaa_condrank_decoding_outputs_spatial_across/full_summary.csv"
)
TEMPORAL_FULL_CSV = Path(
    "msaa_condrank_decoding_outputs_temporal_across/full_summary.csv"
)

OUTPUT_DIR = Path("paper_figure_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_PNG = OUTPUT_DIR / "Figure2_empirical_PCA_MSAA_benchmark_main.png"
FIG_PDF = OUTPUT_DIR / "Figure2_empirical_PCA_MSAA_benchmark_main.pdf"
SOURCE_CSV = OUTPUT_DIR / "Figure2_source_data.csv"

DPI = 300

for p in [RAW_CSV, PCA_CSV, SPATIAL_FULL_CSV, TEMPORAL_FULL_CSV]:
    if not p.exists():
        raise FileNotFoundError(
            f"Missing required input: {p}\n"
            "Run the corresponding upstream analysis notebook first."
        )

In [ ]:
raw = pd.read_csv(RAW_CSV)
pca = pd.read_csv(PCA_CSV)

spatial = pd.read_csv(SPATIAL_FULL_CSV)
spatial["analysis_type"] = "spatial"
if "component_ratio" not in spatial:
    spatial["component_ratio"] = spatial["K"].astype(float) / V

temporal = pd.read_csv(TEMPORAL_FULL_CSV)
temporal["analysis_type"] = "temporal"
if "component_ratio" not in temporal:
    temporal["component_ratio"] = temporal["K"].astype(float) / T

aa = pd.concat([spatial, temporal], ignore_index=True)

if "component_ratio" not in pca:
    pca["component_ratio"] = pca["K"].astype(float) / V

print("Raw rows:", len(raw))
print("PCA rows:", len(pca))
print("AA rows:", len(aa))

In [ ]:
# Export a single combined source-data table for the paper figure.
raw_export = raw.copy()
raw_export["source"] = "raw_htfa"

pca_export = pca.copy()
pca_export["source"] = "pca"

aa_export = aa.copy()
aa_export["source"] = "msaa"

all_cols = sorted(
    set(raw_export.columns)
    | set(pca_export.columns)
    | set(aa_export.columns)
)

pd.concat(
    [
        raw_export.reindex(columns=all_cols),
        pca_export.reindex(columns=all_cols),
        aa_export.reindex(columns=all_cols),
    ],
    ignore_index=True,
).to_csv(SOURCE_CSV, index=False)

print("Saved source data:", SOURCE_CSV)

In [ ]:

from matplotlib.lines import Line2D

# ============================================================
# MAIN-TEXT FIGURE 2
#
# One panel per condition.
# CONDITION = color
# METHOD    = linestyle / marker
# ============================================================

COND_COLORS = {
    "intact": "#7B3294",   # purple
    "word":   "#2CA25F",   # green
    "rest":   "#222222",   # black / dark gray
}

ROW_LABELS = {
    "intact": "Intact narrative",
    "word":   "Word-scrambled",
    "rest":   "Rest",
}

METHOD_STYLES = {
    "pca_reconstruction": {
        "linestyle": "-",
        "marker": "o",
        "markersize": 4.2,
        "linewidth": 2.0,
    },
    "spatial_msaa": {
        "linestyle": "-",
        "marker": "s",
        "markersize": 4.2,
        "linewidth": 2.0,
    },
    "temporal_msaa": {
        "linestyle": "--",
        "marker": "^",
        "markersize": 4.4,
        "linewidth": 2.0,
    },
}

fig, axes = plt.subplots(
    len(CONDITIONS),
    1,
    figsize=(7.4, 10.5),
    sharex=True,
)

panel_letters = ["A", "B", "C"]

for row_i, condition in enumerate(CONDITIONS):

    ax = axes[row_i]
    color = COND_COLORS[condition]

    # --------------------------------------------------------
    # References
    # --------------------------------------------------------
    raw_row = raw[
        raw["condition"].astype(str) == condition
    ]

    raw_value = (
        float(raw_row.iloc[0]["mean"])
        if len(raw_row)
        else np.nan
    )

    chance = 1.0 / T

    # --------------------------------------------------------
    # PCA reconstruction
    # --------------------------------------------------------
    p_rec = (
        pca[
            (pca["condition"].astype(str) == condition)
            & (pca["decode_space"].astype(str) == "reconstruction")
        ]
        .sort_values("component_ratio")
    )

    if len(p_rec):
        ax.errorbar(
            p_rec["component_ratio"],
            p_rec["mean"],
            yerr=(
                p_rec["err"]
                if "err" in p_rec.columns
                else None
            ),
            color=color,
            linestyle=METHOD_STYLES[
                "pca_reconstruction"
            ]["linestyle"],
            marker=METHOD_STYLES[
                "pca_reconstruction"
            ]["marker"],
            markersize=METHOD_STYLES[
                "pca_reconstruction"
            ]["markersize"],
            linewidth=METHOD_STYLES[
                "pca_reconstruction"
            ]["linewidth"],
            capsize=2,
            label="PCA reconstruction",
        )

    # --------------------------------------------------------
    # Spatial MS-AA
    # --------------------------------------------------------
    spatial_cur = (
        aa[
            (aa["analysis_type"].astype(str) == "spatial")
            & (aa["condition"].astype(str) == condition)
        ]
        .sort_values("component_ratio")
    )

    if len(spatial_cur):
        ax.errorbar(
            spatial_cur["component_ratio"],
            spatial_cur["mean"],
            yerr=(
                spatial_cur["err"]
                if "err" in spatial_cur.columns
                else None
            ),
            color=color,
            linestyle=METHOD_STYLES[
                "spatial_msaa"
            ]["linestyle"],
            marker=METHOD_STYLES[
                "spatial_msaa"
            ]["marker"],
            markersize=METHOD_STYLES[
                "spatial_msaa"
            ]["markersize"],
            linewidth=METHOD_STYLES[
                "spatial_msaa"
            ]["linewidth"],
            capsize=2,
            label="Spatial MS-AA",
        )

    # --------------------------------------------------------
    # Temporal MS-AA
    # --------------------------------------------------------
    temporal_cur = (
        aa[
            (aa["analysis_type"].astype(str) == "temporal")
            & (aa["condition"].astype(str) == condition)
        ]
        .sort_values("component_ratio")
    )

    if len(temporal_cur):
        ax.errorbar(
            temporal_cur["component_ratio"],
            temporal_cur["mean"],
            yerr=(
                temporal_cur["err"]
                if "err" in temporal_cur.columns
                else None
            ),
            color=color,
            linestyle=METHOD_STYLES[
                "temporal_msaa"
            ]["linestyle"],
            marker=METHOD_STYLES[
                "temporal_msaa"
            ]["marker"],
            markersize=METHOD_STYLES[
                "temporal_msaa"
            ]["markersize"],
            linewidth=METHOD_STYLES[
                "temporal_msaa"
            ]["linewidth"],
            capsize=2,
            label="Temporal MS-AA",
        )

    # --------------------------------------------------------
    # Raw HTFA + chance
    # --------------------------------------------------------
    if np.isfinite(raw_value):
        ax.axhline(
            raw_value,
            color=color,
            linestyle=":",
            linewidth=1.8,
            alpha=0.95,
            label="Raw HTFA",
        )

    ax.axhline(
        chance,
        color="0.60",
        linestyle=":",
        linewidth=1.2,
        label="Chance",
    )

    # --------------------------------------------------------
    # Limits
    # --------------------------------------------------------
    vals = [chance]

    if np.isfinite(raw_value):
        vals.append(raw_value)

    for df in [
        p_rec,
        spatial_cur,
        temporal_cur,
    ]:
        if len(df):
            vals.extend(
                df["mean"]
                .astype(float)
                .tolist()
            )

    vals = np.asarray(
        [
            v
            for v in vals
            if np.isfinite(v)
        ],
        dtype=float,
    )

    ymin = max(
        0.0,
        vals.min()
        - 0.08 * max(vals.max(), 1e-6),
    )

    ymax = vals.max() * 1.12

    ax.set_ylim(
        ymin,
        ymax,
    )

    ax.set_xlim(
        0,
        1.02,
    )

    # --------------------------------------------------------
    # Row / condition labeling
    # --------------------------------------------------------
    ax.set_ylabel(
        ROW_LABELS[condition],
        fontsize=14,
        fontweight="bold",
        color=color,
        labelpad=42,
    )

    ax.text(
        0.015,
        0.97,
        panel_letters[row_i],
        transform=ax.transAxes,
        fontsize=15,
        fontweight="bold",
        ha="left",
        va="top",
    )

    ax.tick_params(
        axis="both",
        labelsize=11,
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

axes[-1].set_xlabel(
    "Normalized representational ratio",
    fontsize=12,
)

fig.supylabel(
    "Timepoint-identification accuracy",
    fontsize=12,
    x=0.025,
)

# ------------------------------------------------------------
# Shared method legend
# Neutral black handles: condition is represented by row color.
# ------------------------------------------------------------
legend_handles = [
    Line2D(
        [0], [0],
        color="0.2",
        linestyle="-",
        marker="o",
        markersize=5,
        linewidth=2.0,
        label="PCA reconstruction",
    ),
    Line2D(
        [0], [0],
        color="0.2",
        linestyle="-",
        marker="s",
        markersize=5,
        linewidth=2.0,
        label="Spatial MS-AA",
    ),
    Line2D(
        [0], [0],
        color="0.2",
        linestyle="--",
        marker="^",
        markersize=5,
        linewidth=2.0,
        label="Temporal MS-AA",
    ),
    Line2D(
        [0], [0],
        color="0.2",
        linestyle=":",
        linewidth=1.8,
        label="Raw HTFA",
    ),
    Line2D(
        [0], [0],
        color="0.60",
        linestyle=":",
        linewidth=1.2,
        label="Chance",
    ),
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=3,
    frameon=False,
    fontsize=10,
    bbox_to_anchor=(0.5, 0.015),
)

fig.suptitle(
    "Matched PCA vs MS-AA benchmark",
    fontsize=16,
    fontweight="bold",
    y=0.985,
)

fig.subplots_adjust(
    left=0.18,
    right=0.98,
    bottom=0.12,
    top=0.94,
    hspace=0.34,
)

fig.savefig(
    FIG_PNG,
    dpi=DPI,
    bbox_inches="tight",
)

fig.savefig(
    FIG_PDF,
    bbox_inches="tight",
)

plt.show()

print("Saved:", FIG_PNG)
print("Saved:", FIG_PDF)


### Figure 2 interpretation

This main-text figure asks one focused question: how well do PCA reconstruction,
spatial MS-AA, and temporal MS-AA preserve shared timepoint-identifiable
structure as normalized representation size changes?

The projected-space PCA validation is intentionally moved to the supplement.
